## 01b-feature-tag-hierarchy.ipynb

Builds a parent-tag mapping from the MusicBrainz flat tag list.

MusicBrainz has no formal tag taxonomy, so we construct one in two steps:
1. **Substring heuristic** — if tag A contains tag B as a whole-word substring, B is a candidate parent of A (e.g. `death metal` → `metal`)
2. **Manual audit** — the generated `tag_parents.csv` is reviewed and cleaned (codified audit below), then read directly by `02-feature-genre.ipynb` to build the coarse parent-genre rollup

### Outputs
| File | Description |
|---|---|
| `data/tag_parents.csv` | Child → parent mapping for audit/editing |
| `data/tag_parent_counts.csv` | How many albums each parent covers (for prioritising audit) |

In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict

DATA_DIR = '../data'

### 1. Load tags and album tag counts

In [2]:
# All 237k tags with their string names
tags = pd.read_parquet(f'{DATA_DIR}/raw/mb_tag.parquet')
tags['name'] = tags['name'].str.strip().str.lower()
print(f'Total tags: {len(tags):,}')

# Album tag counts — tells us which tags actually appear in the model
album_tags = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
tag_album_counts = album_tags.groupby('tag_id')['album_id'].nunique().rename('n_albums')

# Merge counts onto tag names
tags = tags.merge(tag_album_counts.reset_index(), left_on='id', right_on='tag_id', how='left')
tags['n_albums'] = tags['n_albums'].fillna(0).astype(int)

# Only tags that survive the model's pruning threshold (>=10 albums)
active_tags = tags[tags['n_albums'] >= 10].copy()
print(f'Active tags (>=10 albums): {len(active_tags):,}')
active_tags.sort_values('n_albums', ascending=False).head(20)

Total tags: 237,881
Active tags (>=10 albums): 2,684


,id,name,ref_count,tag_id,n_albums
36715,7,rock,1580049,7.0,268011
36584,11,electronic,1295841,11.0,178655
36717,19,pop,773981,19.0,119598
35913,71,jazz,405871,71.0,93585
36471,166,experimental,306748,166.0,73239
36576,58,ambient,302999,58.0,58832
34761,15,classical,291605,15.0,54684
36723,235,hip hop,366089,235.0,37638
35980,88,punk,267169,88.0,33960
36719,1091,pop rock,205334,1091.0,33291


### 2. Build substring parent map

For each active tag, find all shorter active tags whose name appears as a **whole word** within it.  
Among candidates, pick the **most specific** (longest) parent — so `death metal` maps to `metal`, not `rock`.

In [3]:
# Build set of active tag names sorted by length descending (most specific first)
active_names = active_tags.set_index('name')['n_albums'].to_dict()
sorted_names = sorted(active_names.keys(), key=len, reverse=True)

def find_parent(tag_name, all_names):
    """Return the longest active tag that is a whole-word substring of tag_name,
    excluding the tag itself."""
    for candidate in all_names:
        if candidate == tag_name:
            continue
        if len(candidate) >= len(tag_name):
            continue
        # Whole-word match using word boundaries
        pattern = r'\b' + re.escape(candidate) + r'\b'
        if re.search(pattern, tag_name):
            return candidate
    return None

print('Building parent map (this takes ~1–2 min for the full active tag set)...')

rows = []
for tag_name in sorted_names:
    parent = find_parent(tag_name, sorted_names)
    if parent is not None:
        rows.append({
            'child':         tag_name,
            'parent':        parent,
            'child_albums':  active_names[tag_name],
            'parent_albums': active_names[parent],
        })

parent_map = pd.DataFrame(rows).sort_values('child_albums', ascending=False)
print(f'Child → parent pairs found: {len(parent_map):,}')
parent_map.head(30)

Building parent map (this takes ~1–2 min for the full active tag set)...


Child → parent pairs found: 1,323


,child,parent,child_albums,parent_albums
1257,pop rock,rock,33291,268011
1021,indie rock,indie,30864,5315
340,alternative rock,alternative,29800,2112
1152,synth-pop,synth,19945,76
1149,hard rock,rock,19450,268011
857,black metal,metal,17852,25443
855,heavy metal,metal,17852,25443
341,psychedelic rock,psychedelic,16378,8770
1151,folk rock,folk,16237,30563
741,dark ambient,ambient,14265,58832


### 3. Inspect false positives

Some substring matches are musically wrong — `new wave` → `wave`, `indie pop` → `pop` (debatable), `post-rock` → `rock` (correct).  
Review the top cases before saving.

In [4]:
# Pairs where child has many albums — highest priority to audit
print('=== Top 50 by child album count (audit these first) ===')
parent_map.head(50)[['child', 'parent', 'child_albums', 'parent_albums']]

=== Top 50 by child album count (audit these first) ===


,child,parent,child_albums,parent_albums
1257,pop rock,rock,33291,268011
1021,indie rock,indie,30864,5315
340,alternative rock,alternative,29800,2112
1152,synth-pop,synth,19945,76
1149,hard rock,rock,19450,268011
857,black metal,metal,17852,25443
855,heavy metal,metal,17852,25443
341,psychedelic rock,psychedelic,16378,8770
1151,folk rock,folk,16237,30563
741,dark ambient,ambient,14265,58832


In [5]:
# How many children each parent collects
parent_summary = (
    parent_map.groupby('parent')
    .agg(n_children=('child', 'count'), total_child_albums=('child_albums', 'sum'))
    .sort_values('total_child_albums', ascending=False)
)
print('=== Parents ranked by total child album coverage ===')
parent_summary.head(30)

=== Parents ranked by total child album coverage ===


,n_children,total_child_albums
parent,,
rock,53,96198
metal,40,70404
indie,7,43491
alternative,24,36548
synth,7,22487
folk,20,20407
jazz,27,20250
punk,23,19245
contemporary,15,18987


### 4. Save for manual audit

Edit `tag_parents.csv` directly:
- **Delete rows** where the mapping is wrong (false positives)
- **Edit the `parent` column** to reassign a child to a different parent
- **Add rows** manually for mappings the substring heuristic missed

The `tag_parent_counts.csv` is reference-only — do not edit it.

In [6]:
# Save parent map for audit
out_path = f'{DATA_DIR}/tag_parents.csv'
parent_map[['child', 'parent', 'child_albums', 'parent_albums']].to_csv(out_path, index=False)
print(f'Saved {len(parent_map):,} pairs → {out_path}')

# Save parent summary as reference
ref_path = f'{DATA_DIR}/tag_parent_counts.csv'
parent_summary.reset_index().to_csv(ref_path, index=False)
print(f'Saved parent summary → {ref_path}')

Saved 1,323 pairs → ../data/tag_parents.csv
Saved parent summary → ../data/tag_parent_counts.csv


### 5. Coverage stats

How many albums gain a parent-level signal from this mapping?

In [7]:
# Albums covered by at least one child tag that has a parent
child_tag_ids = (
    tags[tags['name'].isin(parent_map['child'])]['id'].values
)
albums_with_parent = album_tags[album_tags['tag_id'].isin(child_tag_ids)]['album_id'].nunique()
total_tagged = album_tags['album_id'].nunique()

print(f'Albums with at least one child tag : {albums_with_parent:,}')
print(f'Total tagged albums                : {total_tagged:,}')
print(f'Coverage                           : {albums_with_parent/total_tagged*100:.1f}%')

# Distribution of parent genres by album count
parent_name_to_id = tags.set_index('name')['id'].to_dict()
top_parents = parent_summary.head(15).reset_index()
top_parents['parent_id'] = top_parents['parent'].map(parent_name_to_id)
print('\n=== Top 15 parent genres by child album coverage ===')
print(top_parents[['parent', 'n_children', 'total_child_albums']].to_string(index=False))

Albums with at least one child tag : 416,931
Total tagged albums                : 816,390
Coverage                           : 51.1%

=== Top 15 parent genres by child album coverage ===
      parent  n_children  total_child_albums
        rock          53               96198
       metal          40               70404
       indie           7               43491
 alternative          24               36548
       synth           7               22487
        folk          20               20407
        jazz          27               20250
        punk          23               19245
contemporary          15               18987
 psychedelic           7               18010
 progressive          20               17478
     ambient          16               16399
       blues          25               14788
         pop          30               13933
        wave           5               10234


In [8]:
import pandas as pd

DATA_DIR = '../data'
parent_map = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

# Inspect any parent's children
def show_children(parent_name, n=30):
    children = parent_map[parent_map['parent'] == parent_name].sort_values('child_albums', ascending=False)
    print(f'\n=== "{parent_name}" ({len(children)} children) ===')
    print(children[['child', 'child_albums']].head(n).to_string(index=False))

for p in ['wave', 'contemporary', 'synth', 'indie', 'progressive', 'psychedelic', 'ambient']:
    show_children(p)


=== "wave" (5 children) ===


         child  child_albums
      new wave          9487
       no wave           701
     post-wave            23
third wave ska            13
        x-wave            10

=== "contemporary" (15 children) ===
                    child  child_albums
        contemporary jazz         11167
         contemporary r&b          3100
   contemporary christian          1007
       adult contemporary           918
        contemporary folk           814
     contemporary country           792
   contemporary classical           788
      contemporary gospel           177
contemporary instrumental            63
         contemporary r b            42
      contemporary reggae            34
         r b contemporary            29
   contemporary bluegrass            23
       contemporary blues            22
         contemporary pop            11

=== "synth" (7 children) ===
        child  child_albums
    synth-pop         19945
dungeon synth          2009
   synth funk           172
  com

In [9]:
import pandas as pd

DATA_DIR = '../data'
df = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

# 1. Delete bad parents entirely
bad_parents = {'wave', 'dark', 'classic', 'music', 'stoner', 'band',
               'songwriter', 'bass', 'improvisation', 'charts', 'space',
               'synth', 'contemporary', 'garage'}
df = df[~df['parent'].isin(bad_parents)]

# 2. Reassign alternative rock and friends to rock
df.loc[df['child'] == 'alternative rock', 'parent'] = 'rock'

# 3. Reassign progressive sub-genres
prog_to_trance = ['progressive trance', 'progressive psytrance']
prog_to_house  = ['progressive house']
prog_to_country = ['progressive country']
prog_to_delete  = ['progressive breaks', 'progressive soul', 'progressive rap',
                   'progressive-rock', 'progressive-metal']  # hyphen duplicates

df.loc[df['child'].isin(prog_to_trance),  'parent'] = 'trance'
df.loc[df['child'].isin(prog_to_house),   'parent'] = 'house'
df.loc[df['child'].isin(prog_to_country), 'parent'] = 'country'
df = df[~df['child'].isin(prog_to_delete)]

# 4. Reassign psychedelic trance
df.loc[df['child'] == 'psychedelic trance', 'parent'] = 'trance'
df = df[df['child'] != 'psychedelic/garage']

# 5. Remove other malformed/niche tags
df = df[df['child'] != 'rock and indie']
df = df[df['child'] != 'offizielle charts']

# 6. Keep only the valid top-level parent families (final canonical vocabulary).
#    Children whose (possibly reassigned) parent is not one of these are dropped.
VALID_PARENTS = {
    'rock', 'metal', 'indie', 'house', 'jazz', 'pop', 'psychedelic', 'blues', 'ambient', 'punk',
    'progressive', 'hardcore', 'country', 'trance', 'classical', 'alternative', 'dance',
    'experimental', 'noise', 'instrumental',
}
df = df[df['parent'].isin(VALID_PARENTS)]

# 7. Save the canonical cleaned map — consumed directly by 02-feature-genre.ipynb.
df.to_csv(f'{DATA_DIR}/tag_parents.csv', index=False)
print(f'Canonical pairs: {len(df):,}  across {df["parent"].nunique()} parent families')

# Verify remaining parents
print('\nParents (by total child-album coverage):')
print(df.groupby('parent')['child_albums'].sum().sort_values(ascending=False))

Canonical pairs: 401  across 20 parent families

Parents (by total child-album coverage):
parent
rock            125998
metal            70404
indie            42948
jazz             20250
punk             19245
psychedelic      17909
ambient          16399
blues            14788
progressive      14143
pop              13933
house            10327
hardcore          9754
country           9324
classical         7927
alternative       6748
experimental      5795
noise             5420
instrumental      4808
trance            3908
dance             3529
Name: child_albums, dtype: int64
